# 07 — Explainability and Error Analysis

**Objective** — inspect where the selected models fail and which features most affect predictive performance.

**Input** — final grouped train/test split and selected regression/classification models.

**Output** — error tables and permutation-importance summaries for both tasks.

## 1. Rebuild the Final Evaluation Split

In [1]:
from pathlib import Path
import sys
import pandas as pd

current_directory = Path.cwd().resolve()
project_root = next(
    directory for directory in [current_directory, *current_directory.parents]
    if (directory / 'src').is_dir() and (directory / 'notebooks').is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features import prepare_feature_frame, build_preprocessor
from src.modeling import grouped_train_test_split, derive_high_cost_labels, get_regression_candidates, get_classification_candidates
from src.evaluation import regression_error_table, classification_error_table, permutation_importance_table

processed_data_path = project_root / 'data' / 'processed' / 'analytical_dataset.csv'
analytical_dataset = pd.read_csv(processed_data_path)
features, target_charges, customer_groups = prepare_feature_frame(analytical_dataset)
training_features, test_features, training_charges, test_charges, training_groups, test_groups = grouped_train_test_split(
    features, target_charges, customer_groups
)
preprocessor = build_preprocessor(training_features)

## 2. Regression Error Analysis

In [2]:
selected_regression_model = get_regression_candidates(preprocessor)['Random Forest'].fit(
    training_features, training_charges
)
regression_predictions = selected_regression_model.predict(test_features)
regression_error_table(
    pd.DataFrame({'Customer ID': test_groups.values}),
    test_charges.reset_index(drop=True),
    regression_predictions,
).head(10)

,Customer ID,actual,predicted,error,absolute_error
0,Id469,21344.85,2232.815524,-19112.034476,19112.034476
1,Id222,35147.53,18403.723764,-16743.806236,16743.806236
2,Id292,32108.66,15680.659206,-16428.000794,16428.000794
3,Id421,24603.05,8976.898240,-15626.151760,15626.151760
4,Id403,25656.58,10840.859931,-14815.720069,14815.720069
5,Id279,32734.19,18501.940588,-14232.249412,14232.249412
6,Id486,20420.60,6270.162589,-14150.437411,14150.437411
7,Id1822,4646.76,17786.070346,13139.310346,13139.310346
8,Id2097,2302.30,14665.552430,12363.252430,12363.252430
9,Id453,22192.44,10759.822241,-11432.617759,11432.617759


### Regression Permutation Importance

In [3]:
permutation_importance_table(
    selected_regression_model, test_features, test_charges,
    'neg_mean_absolute_error', n_repeats=5
).head(10)

,feature,importance_mean,importance_std
0,smoker,6166.770616,119.229334
1,year,2696.222749,117.772019
2,BMI,2133.487480,104.606813
3,Hospital tier,927.293056,62.878603
4,State ID,446.759368,50.048292
5,children,366.176981,45.585335
6,month,20.056527,16.247309
7,date,7.788128,8.180547
8,Heart Issues,5.812296,3.448617
9,City tier,5.733342,2.740249


## 3. Classification Error Analysis

In [4]:
training_high_cost_labels, test_high_cost_labels, high_cost_threshold = derive_high_cost_labels(
    training_charges, test_charges
)
selected_classifier = get_classification_candidates(preprocessor)['HistGradientBoosting'].fit(
    training_features, training_high_cost_labels
)
classification_probability_scores = selected_classifier.predict_proba(test_features)[:, 1]
classification_predictions = (classification_probability_scores >= 0.5).astype(int)
classification_error_table(
    pd.DataFrame({'Customer ID': test_groups.values}),
    test_high_cost_labels.reset_index(drop=True),
    classification_predictions,
    classification_probability_scores,
).query("error_type != 'correct'").head(10)

,Customer ID,actual,predicted,score,error_type
418,Id280,0,1,0.500159,false_positive
420,Id258,0,1,0.808061,false_positive
421,Id251,0,1,0.689650,false_positive
422,Id240,0,1,0.602009,false_positive
425,Id228,1,0,0.081008,false_negative
428,Id222,1,0,0.000030,false_negative
430,Id215,1,0,0.407347,false_negative
431,Id203,1,0,0.026214,false_negative
438,Id168,1,0,0.452481,false_negative
452,Id94,1,0,0.212436,false_negative


### Classification Permutation Importance

In [5]:
permutation_importance_table(
    selected_classifier, test_features, test_high_cost_labels,
    'average_precision', n_repeats=5
).head(10)

,feature,importance_mean,importance_std
0,smoker,0.787713,0.027750
1,BMI,0.428435,0.085339
2,year,0.088675,0.010923
3,State ID,0.019762,0.005221
4,children,0.019193,0.002538
5,Hospital tier,0.010937,0.003695
6,date,0.004360,0.003385
7,month,0.001807,0.004404
8,City tier,0.000980,0.001507
9,Cancer history,0.000227,0.000748


## Takeaways

Error analysis shows model limits that aggregate metrics can hide. Permutation importance shows how much predictive performance depends on each feature. It does not prove medical causality.